1. 生成all.csv(id, label)

In [8]:
import os
import pandas as pd

folder_path = r"D:\Desktop\iLab\十院\数据集\in_processed"
output_path = r"D:\Desktop\UMKD_NEW\IN\split\all.csv"

data_dict = {}

for filename in os.listdir(folder_path):
    if not os.path.isfile(os.path.join(folder_path, filename)):
        continue

    name, _ = os.path.splitext(filename)
    parts = name.split('-')

    if len(parts) != 3:
        print(f"跳过异常文件名: {filename}")
        continue

    id_, label, mode = parts

    # 只记录一次（因为同一个id有C/G两种）
    if id_ not in data_dict:
        data_dict[id_] = label

# 转成DataFrame
df = pd.DataFrame(list(data_dict.items()), columns=['id', 'label'])

# 保存为CSV
df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"已生成: {output_path}")

已生成: D:\Desktop\UMKD_NEW\IN\split\all.csv


（可选）样本数量

In [7]:
import pandas as pd

file_path = r"D:\Desktop\UMKD_NEW\IN\split\all.csv"

df = pd.read_csv(file_path)

# 统计每个label数量
label_counts = df['label'].value_counts().sort_index()

print("各类别样本数量：")
print(label_counts)

print("\n总样本数：", len(df))

各类别样本数量：
0    466
1    231
2     49
Name: label, dtype: int64

总样本数： 746


2. 生成train_val.csv(id, label)和test.csv(id, label) 9:1

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 读取CSV
input_path = r"D:\Desktop\UMKD_NEW\IN\split\all.csv"
df = pd.read_csv(input_path)

# 分层采样
train_val_df, test_df = train_test_split(
    df,
    test_size=0.1,
    stratify=df['label'],
    random_state=42
)

# 保存结果
test_df.to_csv(r"D:\Desktop\UMKD_NEW\IN\split\test.csv", index=False)
train_val_df.to_csv(r"D:\Desktop\UMKD_NEW\IN\split\train_val.csv", index=False)

print("划分完成：")
print(f"train_val: {len(train_val_df)}")
print(f"test: {len(test_df)}")

划分完成：
train_val: 671
test: 75


(可选)样本数量

In [10]:
import pandas as pd

file_path = r"D:\Desktop\UMKD_NEW\IN\split\train_val.csv"

df = pd.read_csv(file_path)

# 统计每个label数量
label_counts = df['label'].value_counts().sort_index()

print("各类别样本数量：")
print(label_counts)

print("\n总样本数：", len(df))

各类别样本数量：
0    419
1    208
2     44
Name: label, dtype: int64

总样本数： 671


3. 生成ten fold，fold_i.csv

In [11]:
import os
import pandas as pd
from sklearn.model_selection import StratifiedKFold

# 输入输出路径
input_path = r"D:\Desktop\UMKD_NEW\IN\split\train_val.csv"
output_dir = r"D:\Desktop\UMKD_NEW\IN\split\ten_fold"

os.makedirs(output_dir, exist_ok=True)

# 读取数据
df = pd.read_csv(input_path)

# 初始化10折分层
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# 开始划分
for i, (_, val_idx) in enumerate(skf.split(df, df['label'])):
    fold_df = df.iloc[val_idx]

    output_path = os.path.join(output_dir, f"fold_{i}.csv")
    fold_df.to_csv(output_path, index=False)

    print(f"fold_{i} 已保存: {output_path}，样本数: {len(fold_df)}")

print("10-fold划分完成")

fold_0 已保存: D:\Desktop\UMKD_NEW\IN\split\ten_fold\fold_0.csv，样本数: 68
fold_1 已保存: D:\Desktop\UMKD_NEW\IN\split\ten_fold\fold_1.csv，样本数: 67
fold_2 已保存: D:\Desktop\UMKD_NEW\IN\split\ten_fold\fold_2.csv，样本数: 67
fold_3 已保存: D:\Desktop\UMKD_NEW\IN\split\ten_fold\fold_3.csv，样本数: 67
fold_4 已保存: D:\Desktop\UMKD_NEW\IN\split\ten_fold\fold_4.csv，样本数: 67
fold_5 已保存: D:\Desktop\UMKD_NEW\IN\split\ten_fold\fold_5.csv，样本数: 67
fold_6 已保存: D:\Desktop\UMKD_NEW\IN\split\ten_fold\fold_6.csv，样本数: 67
fold_7 已保存: D:\Desktop\UMKD_NEW\IN\split\ten_fold\fold_7.csv，样本数: 67
fold_8 已保存: D:\Desktop\UMKD_NEW\IN\split\ten_fold\fold_8.csv，样本数: 67
fold_9 已保存: D:\Desktop\UMKD_NEW\IN\split\ten_fold\fold_9.csv，样本数: 67
10-fold划分完成


4. 生成ten fold balance，fold_i.csv
419, 208, 44 -> 208, 208, 44

In [13]:
import pandas as pd

input_path = r"D:\Desktop\UMKD_NEW\IN\split\train_val.csv"
output_path = r"D:\Desktop\UMKD_NEW\IN\split\train_val_balance.csv"

df = pd.read_csv(input_path)

print("原始分布：")
print(df['label'].value_counts())

# 按你的设定（如果label不是0/1/2，请先检查）
A = df[df['label'] == 0]
B = df[df['label'] == 1]
C = df[df['label'] == 2]

# 目标（只下采样，不上采样）
target_A = 208

A_bal = A.sample(n=target_A, random_state=42)
B_bal = B.copy()
C_bal = C.copy()

df_bal = pd.concat([A_bal, B_bal, C_bal])

# shuffle
df_bal = df_bal.sample(frac=1, random_state=42).reset_index(drop=True)

print("\n新分布：")
print(df_bal['label'].value_counts())

df_bal.to_csv(output_path, index=False)

print("已保存：", output_path)

原始分布：
0    419
1    208
2     44
Name: label, dtype: int64

新分布：
0    208
1    208
2     44
Name: label, dtype: int64
已保存： D:\Desktop\UMKD_NEW\IN\split\train_val_balance.csv


In [14]:
import os
import pandas as pd
from sklearn.model_selection import StratifiedKFold

input_path = r"D:\Desktop\UMKD_NEW\IN\split\train_val_balance.csv"
output_dir = r"D:\Desktop\UMKD_NEW\IN\split\ten_fold_balance"

os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(input_path)

print("数据分布：")
print(df['label'].value_counts())

# 10-fold分层
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

for i, (_, val_idx) in enumerate(skf.split(df, df['label'])):
    fold_df = df.iloc[val_idx].copy()

    save_path = os.path.join(output_dir, f"fold_{i}.csv")
    fold_df.to_csv(save_path, index=False)

    print(f"\nfold_{i}：")
    print(fold_df['label'].value_counts())

print("\n10-fold划分完成")

数据分布：
0    208
1    208
2     44
Name: label, dtype: int64

fold_0：
0    21
1    20
2     5
Name: label, dtype: int64

fold_1：
0    21
1    20
2     5
Name: label, dtype: int64

fold_2：
1    21
0    21
2     4
Name: label, dtype: int64

fold_3：
0    21
1    21
2     4
Name: label, dtype: int64

fold_4：
0    21
1    21
2     4
Name: label, dtype: int64

fold_5：
1    21
0    21
2     4
Name: label, dtype: int64

fold_6：
0    21
1    21
2     4
Name: label, dtype: int64

fold_7：
0    21
1    21
2     4
Name: label, dtype: int64

fold_8：
1    21
0    20
2     5
Name: label, dtype: int64

fold_9：
1    21
0    20
2     5
Name: label, dtype: int64

10-fold划分完成
